In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
# Install required packages
!uv pip install langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu numpy python-dotenv rank-bm25 pymupdf pypdf chromadb

Using Python 3.12.13 environment at: /usr
Resolved 117 packages in 1.95s
Prepared 25 packages in 2.74s
Uninstalled 2 packages in 34ms
Installed 25 packages in 215ms
 + bcrypt==5.0.0
 + build==1.5.0
 + chromadb==1.5.9
 + dataclasses-json==0.6.7
 + durationpy==0.10
 + faiss-cpu==1.13.2
 + kubernetes==35.0.0
 + langchain-classic==1.0.7
 + langchain-community==0.4.1
 - langchain-core==1.3.1
 + langchain-core==1.3.3
 + langchain-openai==1.2.1
 + langchain-protocol==0.0.15
 + langchain-text-splitters==1.1.2
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0
 + onnxruntime==1.25.1
 + opentelemetry-exporter-otlp-proto-grpc==1.38.0
 + pybase64==1.4.3
 + pymupdf==1.27.2.3
 + pypdf==6.10.2
 + pypika==0.51.1
 + pyproject-hooks==1.2.0
 + rank-bm25==0.2.2
 - requests==2.32.4
 + requests==2.33.1
 + typing-inspect==0.9.0


In [ ]:
!uv pip install numpy rank-bm25 pymupdf deepeval

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 450ms
Prepared 10 packages in 227ms
Installed 10 packages in 59ms
 + backoff==2.2.1
 + deepeval==3.9.9
 + execnet==2.1.2
 + portalocker==3.2.0
 + posthog==7.14.0
 + pyfiglet==1.0.4
 + pytest-asyncio==1.3.0
 + pytest-repeat==0.9.4
 + pytest-rerunfailures==16.1
 + pytest-xdist==3.8.0


In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np

sys.path.append('/content/drive/MyDrive/Datasets')

# Load environment variables from a .env file
load_dotenv("/content/drive/MyDrive/Datasets/API_KEYS.env")

# Set the OpenAI API key environment variable
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

from utils.helper_functions import *
from utils.evaluate_rag import *

In [ ]:
path = "/content/drive/MyDrive/Datasets/pdfs/Understanding_Climate_Change.pdf"

In [ ]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A tuple of (FAISS vector store, cleaned text documents).
    """
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS

    # Load PDF Documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documetns into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, chunk_overlap = chunk_overlap, length_function = len
    )

    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

## Create vectorstore and get chunked documents

In [ ]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

## Create a bm25 index for retrieving documents by keywords

In [ ]:
from rank_bm25 import BM25Okapi

# Initializing
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus = [doc.split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

# Ranking of documents
query = "windy london"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)
print(doc_scores)

doc_top_n = bm25.get_top_n(tokenized_query, corpus, n=1)
print(doc_top_n)

[0.         0.46864736 0.        ]
['It is quite windy in London']


In [ ]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [ ]:
bm25 = create_bm25_index(cleaned_texts)

### Define a function that retrieves both semantically and by keyword, normalizes the scores and gets the top k documents

In [ ]:
def fusion_retrieval(vectorstore, bm25, query:str, k:int=5, alpha:float=0.5) -> List[Document]:
    """
    Perform fusion retrieval combining keyword-based (BM25) and vector-based search.

    Args:
    vectorstore (VectorStore): The vectorstore containing the documents.
    bm25 (BM25Okapi): Pre-computed BM25 index.
    query (str): The query string.
    k (int): The number of documents to retrieve.
    alpha (float): The weight for vector search scores (1-alpha will be the weight for BM25 scores).

    Returns:
    List[Document]: The top k documents based on the combined scores.
    """
    epsilon = 1e-8

    # Step 1: Get all documents from the vectorstore
    all_docs = vectorstore.similarity_search("", k=vectorstore.index.ntotal)

    # Step 2: Perform BM25 search
    bm25_scores = bm25.get_scores(query.split())

    # Step 3: Perform vector search
    vector_results = vectorstore.similarity_search_with_score(query, k=len(all_docs))

    # Step 4: Normalize scores
    vector_scores = np.array([score for _, score in vector_results])
    vector_scores = 1 - (vector_scores - np.max(bm25_scores))/(np.max(vector_scores) - np.min(vector_scores)+epsilon)

    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores)+epsilon)

    # Step 5: Combine scores
    combined_scores = alpha * vector_scores + (1-alpha)*bm25_scores

    # Step 6: Rank documents
    sorted_indices = np.argsort(combined_scores)[::-1]

    # Step 7: Return top k documents
    return [all_docs[i] for i in sorted_indices[:k]]

## Use Case example

In [ ]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
top_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.8)
docs_content = [doc.page_content for doc in top_docs]
show_context(docs_content)

Context 1:
Vision for a Sustainable Future 
Holistic Approach 
Addressing climate change requires a holistic approach that integrates environmental, social, 
and economic dimensions. Sustainable development, circular economy, and ecological justice 
are key principles guiding this approach. Collaboration across sectors and scales is essential 
for achieving a sustainable future. 
Innovation and Creativity 
Innovation and creativity are vital for developing new solutions to climate challenges. This 
includes technological advancements, policy innovations, and creative approaches to 
education and communication. Fostering a culture of innovation supports continuous 
improvement and adaptation. 
Global Solidarity 
Global solidarity and cooperation are fundamental for addressing the global challenge of 
climate change. This includes supporting vulnerable countries and communities, sharing 
resources and technologies, and promoting equitable solutions. Solidarity strengthens global


Contex

## Langchain Implementation

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Load PDF
loader = PyPDFLoader('/content/drive/MyDrive/Datasets/pdfs/Understanding_Climate_Change.pdf')
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

# Dense retriever
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(splits, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# Sparse retriever
sparse_retriever = BM25Retriever.from_documents(splits)
sparse_retriever.k=3

alpha:float=0.5

# Hybrid retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[alpha, 1-alpha]
)

# LLM
llm = ChatOpenAI(model='gpt-4', temperature=0)

# Create RAG Chain
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = hybrid_retriever,
    return_source_documents = True
)

# Query with generation
query = "What is the main topic of this document?"
result = qa_chain.invoke({"query":query})

print("Answer:", result['result'])
print("\nSources:")
for doc in result["source_documents"]:
  print(f"-{doc.page_content[:200]}...")

Answer: The main topic of this document is climate change and the various strategies and challenges associated with addressing it, including policy, technology, and global cooperation.

Sources:
-challenges. This vision includes healthy ecosystems, sustainable economies, and social 
justice. Achieving this vision requires commitment, innovation, and collective effort. 
Legacy for Future Genera...
-emissions, particularly methane, which is a potent greenhouse gas. Innovations in fracking 
technology have made natural gas more accessible, but this comes with environmental and 
health concerns. 
D...
-Direct Air Capture (DAC) 
DAC technology removes CO2 directly from the atmosphere, offering a way to achieve 
negative emissions. The captured CO2 can be stored or used in various applications. Scalin...
-for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning 

# RRF

In [ ]:
!pip install scikit-learn sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def reciprocal_rank_fusion(rankings, k=60, weights=None):
    """RRF implementation"""
    if weights is None:
        weights = [1.0] * len(rankings)

    doc_scores = {}
    for ranking_idx, ranking in enumerate(rankings):
        weight = weights[ranking_idx]
        for rank_position, doc_id in enumerate(ranking, start=1):
            rrf_score = weight * (1.0 / (k + rank_position))
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + rrf_score

    return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = "What is AI"

# ============================================
# DENSE SEARCH (Vector/Semantic)
# ============================================
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Calculate cosine similarity
similarities = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Get ranked document IDs (by similarity)
dense_ranking = np.argsort(similarities)[::-1].tolist()

# ============================================
# SPARSE SEARCH (Keyword/BM25-like)
# ============================================
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
tfidf_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# Get ranked document IDs
sparse_ranking = np.argsort(tfidf_scores)[::-1].tolist()

# ============================================
# HYBRID SEARCH WITH RRF
# ============================================
hybrid_results = reciprocal_rank_fusion(
    rankings=[dense_ranking, sparse_ranking],
    k=60,
    weights=[2.0, 1.0]  # Dense search more important
)

# ============================================
# DISPLAY RESULTS
# ============================================
print("QUERY:", query)
print("\n" + "="*60)
print("DENSE SEARCH RANKING:")
for rank, doc_idx in enumerate(dense_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("SPARSE SEARCH RANKING:")
for rank, doc_idx in enumerate(sparse_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("HYBRID RRF RESULTS:")
print("="*60)
for doc_idx, score in hybrid_results[:3]:  # Top 3
    print(f"Score: {score:.6f}")
    print(f"Doc:   {documents[doc_idx]}")
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

QUERY: What is AI

DENSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [4] Computer vision enables machines to interpret images
3. [3] Natural language processing helps computers understand text
4. [1] Deep learning uses neural networks with multiple layers
5. [2] Python is popular for data science and ML

SPARSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [2] Python is popular for data science and ML
3. [4] Computer vision enables machines to interpret images
4. [3] Natural language processing helps computers understand text
5. [1] Deep learning uses neural networks with multiple layers

HYBRID RRF RESULTS:
Score: 0.049180
Doc:   Machine learning is a subset of artificial intelligence

Score: 0.048131
Doc:   Computer vision enables machines to interpret images

Score: 0.047371
Doc:   Natural language processing helps computers understand text



In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def relative_score_fusion(score_lists, weights=None):
    """
    RSF implementation - uses actual scores, not just ranks

    Args:
        score_lists: List of lists, each containing (doc_id, score) tuples
        weights: Optional weights for each retriever
    """
    if weights is None:
      weights = [1.0]*len(score_lists)
    doc_scores={}
    for retriever_idx, scores in enumerate(score_lists):
      weight = weights[retriever_idx]
      # Extract just scores for normalization
      raw_scores = [score for _, score in scores]

      if len(raw_scores) ==0:
        continue
      min_score = min(raw_scores)
      max_score = max(raw_scores)
      score_range = max_score - min_score

      # Normalize and accumulate
      for doc_id, score in scores:
        if score_range == 0:
          normalized = 1.0
        else:
          normalized = (score - min_score)/score_range

        doc_scores[doc_id]=doc_scores.get(doc_id, 0.0)+(weight*normalized)
    return sorted(doc_scores.items(), key=lambda x:x[1], reverse=True)

# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = 'What is AI?'

# DENSE SEARCH
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Claculate cosine similarity
dense_scores = np.dot(doc_embeddings, query_embedding)/(
    np.linalg.norm(doc_embeddings, axis=1)*np.linalg.norm(query_embedding)
)

# Create (doc_id, score) tuples - sorted by score descending
dense_with_scores = [(i, float(dense_scores[i])) for i in range(len(documents))]
dense_with_scores.sort(key=lambda x:x[1], reverse = True)

# SPARSE SEARCH
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
sparse_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# create (doc_id, score) tuples - sorted by score descending
sparse_with_scores = [(i, float(sparse_scores[i])) for i in range(len(documents))]
sparse_with_scores.sort(key=lambda x:x[1], reverse=True)

# HYBRID SEARCH WITH RSF
hybrid_results = relative_score_fusion(
    score_lists=[dense_with_scores, sparse_with_scores],
    weights=[2.0, 1.0]
)

# DISPLAY RESULTS
print("QUERY:", query)

print("\n"+"="*60)
print("DENSE SEARCH(with raw scores):")
print("="*60)
for doc_idx, score in dense_with_scores:
  print(f"Score:{score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n"+"="*60)
print("SPARSE SEARCH(with raw scores):")
print("="*60)
for doc_idx, score in sparse_with_scores:
  print(f"Score:{score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n"+"="*60)
print("HYBRID RSF RESULTS:")
print("="*60)

# Show normalization breakdown for top results
print("\nDetailed scoring breakdown:\n")

# Recalculate normalized scores for display
dense_raw = [s for _, s in dense_with_scores]
sparse_raw = [s for _, s in sparse_with_scores]

dense_min, dense_max = min(dense_raw), max(dense_raw)
sparse_min, sparse_max = min(sparse_raw), max(sparse_raw)

for doc_idx, final_score in hybrid_results[:3]:
  dense_score = dense_scores[doc_idx]
  sparse_score = sparse_scores[doc_idx]

  # Normalized scores
  dense_norm = (dense_score - dense_min)/(dense_max - dense_min)
  sparse_norm = (sparse_score - sparse_min)/(sparse_max - sparse_min) if (sparse_max - sparse_min)>0 else 0

  print(f"Doc [{doc_idx}]:{documents[doc_idx]}")
  print(f"  Dense:  raw={dense_score:.4f} -> norm={dense_norm:.4f} × weight=2.0 = {dense_norm*2:.4f}")
  print(f"  Sparse: raw={sparse_score:.4f} -> norm={sparse_norm:.4f} × weight=2.0 = {sparse_norm*2:.4f}")
  print(f"  Final RSF score:{final_score:.4f}")
  print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: What is AI?

DENSE SEARCH(with raw scores):
Score:0.5690 | [0] Machine learning is a subset of artificial intelligence
Score:0.3552 | [4] Computer vision enables machines to interpret images
Score:0.3396 | [3] Natural language processing helps computers understand text
Score:0.2673 | [1] Deep learning uses neural networks with multiple layers
Score:0.2498 | [2] Python is popular for data science and ML

SPARSE SEARCH(with raw scores):
Score:0.3214 | [0] Machine learning is a subset of artificial intelligence
Score:0.2917 | [2] Python is popular for data science and ML
Score:0.0000 | [1] Deep learning uses neural networks with multiple layers
Score:0.0000 | [3] Natural language processing helps computers understand text
Score:0.0000 | [4] Computer vision enables machines to interpret images

HYBRID RSF RESULTS:

Detailed scoring breakdown:

Doc [0]:Machine learning is a subset of artificial intelligence
  Dense:  raw=0.5690 -> norm=1.0000 × weight=2.0 = 2.0000
  Sparse: raw=0.321